# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [16]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [17]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [18]:
spaceship_model = spaceship.copy()

spending_columns = [
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck"
]

spaceship_model["TotalSpending"] = spaceship_model[spending_columns].sum(axis=1)
spaceship_model["NoSpending"] = spaceship_model["TotalSpending"] == 0

cabin_parts = spaceship_model["Cabin"].str.split("/", expand=True)
spaceship_model["Deck"] = cabin_parts[0]
spaceship_model["CabinNumber"] = pd.to_numeric(cabin_parts[1], errors="coerce")
spaceship_model["Side"] = cabin_parts[2]

X = spaceship_model.drop(
    columns=["Transported", "PassengerId", "Cabin", "Name"]
)

y = spaceship_model["Transported"].astype(int)

print("Selected features:", list(X.columns))

Selected features: ['HomePlanet', 'CryoSleep', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'TotalSpending', 'NoSpending', 'Deck', 'CabinNumber', 'Side']


**Perform Train Test Split**

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_columns = X.select_dtypes(include="number").columns
categorical_columns = X.select_dtypes(include=["object", "bool"]).columns

numeric_preprocessing = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessing = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_preprocessing, numeric_columns),
    ("categorical", categorical_preprocessing, categorical_columns)
])

In [21]:
results = []

In [10]:
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

bagging_model = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=100,
        max_samples=0.8,
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    ))
])

bagging_model.fit(X_train, y_train)
bagging_predictions = bagging_model.predict(X_test)
bagging_probabilities = bagging_model.predict_proba(X_test)[:, 1]

results.append({
    "model": "Bagging",
    "accuracy": accuracy_score(y_test, bagging_predictions),
    "precision": precision_score(y_test, bagging_predictions),
    "recall": recall_score(y_test, bagging_predictions),
    "f1": f1_score(y_test, bagging_predictions),
    "roc_auc": roc_auc_score(y_test, bagging_probabilities)
})

In [11]:
pasting_model = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=42),
        n_estimators=100,
        max_samples=0.8,
        bootstrap=False,
        random_state=42,
        n_jobs=-1
    ))
])

pasting_model.fit(X_train, y_train)
pasting_predictions = pasting_model.predict(X_test)
pasting_probabilities = pasting_model.predict_proba(X_test)[:, 1]

results.append({
    "model": "Pasting",
    "accuracy": accuracy_score(y_test, pasting_predictions),
    "precision": precision_score(y_test, pasting_predictions),
    "recall": recall_score(y_test, pasting_predictions),
    "f1": f1_score(y_test, pasting_predictions),
    "roc_auc": roc_auc_score(y_test, pasting_probabilities)
})

- Random Forests

In [12]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("model", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

random_forest_model.fit(X_train, y_train)
rf_predictions = random_forest_model.predict(X_test)
rf_probabilities = random_forest_model.predict_proba(X_test)[:, 1]

results.append({
    "model": "Random Forest",
    "accuracy": accuracy_score(y_test, rf_predictions),
    "precision": precision_score(y_test, rf_predictions),
    "recall": recall_score(y_test, rf_predictions),
    "f1": f1_score(y_test, rf_predictions),
    "roc_auc": roc_auc_score(y_test, rf_probabilities)
})

- Gradient Boosting

In [13]:
from sklearn.ensemble import GradientBoostingClassifier

gradient_boosting_model = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("model", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ))
])

gradient_boosting_model.fit(X_train, y_train)
gb_predictions = gradient_boosting_model.predict(X_test)
gb_probabilities = gradient_boosting_model.predict_proba(X_test)[:, 1]

results.append({
    "model": "Gradient Boosting",
    "accuracy": accuracy_score(y_test, gb_predictions),
    "precision": precision_score(y_test, gb_predictions),
    "recall": recall_score(y_test, gb_predictions),
    "f1": f1_score(y_test, gb_predictions),
    "roc_auc": roc_auc_score(y_test, gb_probabilities)
})

- Adaptive Boosting

In [14]:
from sklearn.ensemble import AdaBoostClassifier

adaboost_model = Pipeline([
    ("preprocessing", clone(preprocessor)),
    ("model", AdaBoostClassifier(
        n_estimators=100,
        learning_rate=0.5,
        random_state=42
    ))
])

adaboost_model.fit(X_train, y_train)
ada_predictions = adaboost_model.predict(X_test)
ada_probabilities = adaboost_model.predict_proba(X_test)[:, 1]

results.append({
    "model": "AdaBoost",
    "accuracy": accuracy_score(y_test, ada_predictions),
    "precision": precision_score(y_test, ada_predictions),
    "recall": recall_score(y_test, ada_predictions),
    "f1": f1_score(y_test, ada_predictions),
    "roc_auc": roc_auc_score(y_test, ada_probabilities)
})

Which model is the best and why?

In [15]:
results_df = pd.DataFrame(results).set_index("model")
results_df.sort_values("f1", ascending=False).round(3)

,accuracy,precision,recall,f1,roc_auc
model,,,,,
Gradient Boosting,0.802,0.796,0.816,0.806,0.903
Bagging,0.807,0.831,0.775,0.802,0.900
Random Forest,0.806,0.836,0.765,0.799,0.893
Pasting,0.803,0.826,0.772,0.798,0.887
AdaBoost,0.769,0.796,0.727,0.760,0.857


### Conclusion

Gradient Boosting performed best when comparing F1 score (0.806) and ROC AUC (0.903), and it also had the highest recall (0.816). Bagging had the highest accuracy (0.807), while Random Forest had the highest precision (0.836). The scores are close, so there is no single winner across every metric. Based on the balance of precision and recall, Gradient Boosting is the strongest overall choice in this test split.

These results come from one train-test split, so cross-validation would give a more reliable comparison. The previous KNN notebook used a different feature set and preprocessing, so its accuracy should not be compared directly with these ensemble results.
